In [1]:
!nvidia-smi

Thu Jun 19 13:18:24 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.154.05             Driver Version: 535.154.05   CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA H100 80GB HBM3          On  | 00000000:19:00.0 Off |                    0 |
| N/A   45C    P0             459W / 700W |  60173MiB / 81559MiB |    100%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [2]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [3]:
from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoConfig,
    AutoProcessor,
    AutoTokenizer,
    # CLIPVisionConfig,
    # LlamaConfig,
    RobertaConfig,
)

import torch
import torch.nn as nn

from modeling_llava_code import LlavaCodeConfig,  LlavaCodeForConditionalGeneration

from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()
token = os.getenv("HF_TOKEN")
login(token=token)

device = torch.device("cuda:0")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
from dataclasses import dataclass

@dataclass
class ArgsMock:
    text_model_id = "bigcode/starcoderbase-1b"
    structure_model_id = "microsoft/unixcoder-base"
    # model_name_or_path = 'unsloth/Llama-3.2-1B'

args = ArgsMock()

In [5]:
structure_config = RobertaConfig.from_pretrained(args.structure_model_id)
structure_config.model_id = args.structure_model_id
text_config = AutoConfig.from_pretrained(args.text_model_id)
configuration = LlavaCodeConfig(structure_config, text_config)
configuration

LlavaCodeConfig {
  "model_type": "llava_next",
  "multimodal_projector_bias": true,
  "projector_hidden_act": "gelu",
  "structure_config": {
    "architectures": [
      "RobertaModel"
    ],
    "attention_probs_dropout_prob": 0.1,
    "classifier_dropout": null,
    "gradient_checkpointing": false,
    "hidden_act": "gelu",
    "hidden_dropout_prob": 0.1,
    "hidden_size": 768,
    "initializer_range": 0.02,
    "intermediate_size": 3072,
    "layer_norm_eps": 1e-05,
    "max_position_embeddings": 1026,
    "model_id": "microsoft/unixcoder-base",
    "model_type": "roberta",
    "num_attention_heads": 12,
    "num_hidden_layers": 12,
    "output_past": true,
    "position_embedding_type": "absolute",
    "torch_dtype": "float32",
    "type_vocab_size": 10,
    "use_cache": true,
    "vocab_size": 51416
  },
  "text_config": {
    "_name_or_path": "bigcode/starcoderbase-1b",
    "activation_function": "gelu_pytorch_tanh",
    "architectures": [
      "GPTBigCodeForCausalLM"
    ],


In [6]:
structure_config

RobertaConfig {
  "architectures": [
    "RobertaModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 1026,
  "model_id": "microsoft/unixcoder-base",
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.51.3",
  "type_vocab_size": 10,
  "use_cache": true,
  "vocab_size": 51416
}

In [7]:
text_config

GPTBigCodeConfig {
  "activation_function": "gelu_pytorch_tanh",
  "architectures": [
    "GPTBigCodeForCausalLM"
  ],
  "attention_softmax_in_fp32": true,
  "attn_pdrop": 0.1,
  "bos_token_id": 0,
  "embd_pdrop": 0.1,
  "eos_token_id": 0,
  "inference_runner": 0,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "max_batch_size": null,
  "max_sequence_length": null,
  "model_type": "gpt_bigcode",
  "multi_query": true,
  "n_embd": 2048,
  "n_head": 16,
  "n_inner": 8192,
  "n_layer": 24,
  "n_positions": 8192,
  "pad_key_length": true,
  "pre_allocate_kv_cache": false,
  "resid_pdrop": 0.1,
  "scale_attention_softmax_in_fp32": true,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "torch_dtype": "float32",
  "transformers_version": "4.51.3",
  "use_cache": true,
  "validate_runner_input": true,
  "vocab_size": 49152
}

In [8]:
configuration.output_attentions, configuration.output_hidden_states, configuration.use_return_dict

(False, False, True)

In [9]:
model = LlavaCodeForConditionalGeneration(configuration).to(device)

In [10]:
model

LlavaCodeForConditionalGeneration(
  (model): LlavaCodeModel(
    (structure_model): UniXcoder(
      (model): RobertaModel(
        (embeddings): RobertaEmbeddings(
          (word_embeddings): Embedding(51416, 768, padding_idx=1)
          (position_embeddings): Embedding(1026, 768, padding_idx=1)
          (token_type_embeddings): Embedding(10, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): RobertaEncoder(
          (layer): ModuleList(
            (0-11): 12 x RobertaLayer(
              (attention): RobertaAttention(
                (self): RobertaSdpaSelfAttention(
                  (query): Linear(in_features=768, out_features=768, bias=True)
                  (key): Linear(in_features=768, out_features=768, bias=True)
                  (value): Linear(in_features=768, out_features=768, bias=True)
                  (dropout): Dropout(p=0.1, inplace=False)
          

In [11]:
processor = AutoProcessor.from_pretrained(args.text_model_id)
prompt = 'with open('
inputs = processor(prompt, return_tensors="pt").to(device)
inputs

{'input_ids': tensor([[1793, 2156,   26]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1]], device='cuda:0')}

In [12]:
output = model.generate(**inputs, max_new_tokens=5)

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Args: torch.Size([1, 3]) torch.Size([1, 3]) torch.Size([1, 3, 2048])
Cache: True hidden states: False output attentions False
past_key_values <class 'transformers.cache_utils.DynamicCache'>
kwargs: {}
last hidden state shape torch.Size([1, 3, 2048])
torch.Size([1, 3, 2048])

Args: torch.Size([1, 4]) torch.Size([1, 1]) torch.Size([1, 1, 2048])
Cache: True hidden states: False output attentions False
past_key_values <class 'list'>
kwargs: {}
last hidden state shape torch.Size([1, 1, 2048])
torch.Size([1, 1, 2048])

Args: torch.Size([1, 5]) torch.Size([1, 1]) torch.Size([1, 1, 2048])
Cache: True hidden states: False output attentions False
past_key_values <class 'list'>
kwargs: {}
last hidden state shape torch.Size([1, 1, 2048])
torch.Size([1, 1, 2048])

Args: torch.Size([1, 6]) torch.Size([1, 1]) torch.Size([1, 1, 2048])
Cache: True hidden states: False output attentions False
past_key_values <class 'list'>
kwargs: {}
last hidden state shape torch.Size([1, 1, 2048])
torch.Size([1, 1, 204

In [13]:
output

tensor([[1793, 2156,   26, 4887, 4887, 7218, 7218, 7218]], device='cuda:0')

In [14]:
model

LlavaCodeForConditionalGeneration(
  (model): LlavaCodeModel(
    (structure_model): UniXcoder(
      (model): RobertaModel(
        (embeddings): RobertaEmbeddings(
          (word_embeddings): Embedding(51416, 768, padding_idx=1)
          (position_embeddings): Embedding(1026, 768, padding_idx=1)
          (token_type_embeddings): Embedding(10, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): RobertaEncoder(
          (layer): ModuleList(
            (0-11): 12 x RobertaLayer(
              (attention): RobertaAttention(
                (self): RobertaSdpaSelfAttention(
                  (query): Linear(in_features=768, out_features=768, bias=True)
                  (key): Linear(in_features=768, out_features=768, bias=True)
                  (value): Linear(in_features=768, out_features=768, bias=True)
                  (dropout): Dropout(p=0.1, inplace=False)
          

In [15]:
processor.decode(output[0])

'with open(velopmentvelopmentMovMovMov'

In [16]:
state_dict = AutoModelForCausalLM.from_pretrained(args.text_model_id).state_dict()
state_dict = {key.replace('transformer.', 'model.language_model.'): value for (key, value) in state_dict.items()}
model.load_state_dict(state_dict, strict=False)

_IncompatibleKeys(missing_keys=['model.image_newline', 'model.structure_model.bias', 'model.structure_model.model.embeddings.word_embeddings.weight', 'model.structure_model.model.embeddings.position_embeddings.weight', 'model.structure_model.model.embeddings.token_type_embeddings.weight', 'model.structure_model.model.embeddings.LayerNorm.weight', 'model.structure_model.model.embeddings.LayerNorm.bias', 'model.structure_model.model.encoder.layer.0.attention.self.query.weight', 'model.structure_model.model.encoder.layer.0.attention.self.query.bias', 'model.structure_model.model.encoder.layer.0.attention.self.key.weight', 'model.structure_model.model.encoder.layer.0.attention.self.key.bias', 'model.structure_model.model.encoder.layer.0.attention.self.value.weight', 'model.structure_model.model.encoder.layer.0.attention.self.value.bias', 'model.structure_model.model.encoder.layer.0.attention.output.dense.weight', 'model.structure_model.model.encoder.layer.0.attention.output.dense.bias', 'm

In [17]:
output = model.generate(**inputs, max_new_tokens=5)

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Args: torch.Size([1, 3]) torch.Size([1, 3]) torch.Size([1, 3, 2048])
Cache: True hidden states: False output attentions False
past_key_values <class 'transformers.cache_utils.DynamicCache'>
kwargs: {}
last hidden state shape torch.Size([1, 3, 2048])
torch.Size([1, 3, 2048])

Args: torch.Size([1, 4]) torch.Size([1, 1]) torch.Size([1, 1, 2048])
Cache: True hidden states: False output attentions False
past_key_values <class 'list'>
kwargs: {}
last hidden state shape torch.Size([1, 1, 2048])
torch.Size([1, 1, 2048])

Args: torch.Size([1, 5]) torch.Size([1, 1]) torch.Size([1, 1, 2048])
Cache: True hidden states: False output attentions False
past_key_values <class 'list'>
kwargs: {}
last hidden state shape torch.Size([1, 1, 2048])
torch.Size([1, 1, 2048])

Args: torch.Size([1, 6]) torch.Size([1, 1]) torch.Size([1, 1, 2048])
Cache: True hidden states: False output attentions False
past_key_values <class 'list'>
kwargs: {}
last hidden state shape torch.Size([1, 1, 2048])
torch.Size([1, 1, 204

In [18]:
processor.decode(output[0])

'with open(os.path.join'

In [19]:
model.language_model

GPTBigCodeModel(
  (wte): Embedding(49152, 2048)
  (wpe): Embedding(8192, 2048)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-23): 24 x GPTBigCodeBlock(
      (ln_1): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (attn): GPTBigCodeSdpaAttention(
        (c_attn): Linear(in_features=2048, out_features=2304, bias=True)
        (c_proj): Linear(in_features=2048, out_features=2048, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (mlp): GPTBigCodeMLP(
        (c_fc): Linear(in_features=2048, out_features=8192, bias=True)
        (c_proj): Linear(in_features=8192, out_features=2048, bias=True)
        (act): PytorchGELUTanh()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
)

In [20]:
for name, param in model.named_parameters():
    print(name)

model.image_newline
model.structure_model.model.embeddings.word_embeddings.weight
model.structure_model.model.embeddings.position_embeddings.weight
model.structure_model.model.embeddings.token_type_embeddings.weight
model.structure_model.model.embeddings.LayerNorm.weight
model.structure_model.model.embeddings.LayerNorm.bias
model.structure_model.model.encoder.layer.0.attention.self.query.weight
model.structure_model.model.encoder.layer.0.attention.self.query.bias
model.structure_model.model.encoder.layer.0.attention.self.key.weight
model.structure_model.model.encoder.layer.0.attention.self.key.bias
model.structure_model.model.encoder.layer.0.attention.self.value.weight
model.structure_model.model.encoder.layer.0.attention.self.value.bias
model.structure_model.model.encoder.layer.0.attention.output.dense.weight
model.structure_model.model.encoder.layer.0.attention.output.dense.bias
model.structure_model.model.encoder.layer.0.attention.output.LayerNorm.weight
model.structure_model.model.

In [21]:
model

LlavaCodeForConditionalGeneration(
  (model): LlavaCodeModel(
    (structure_model): UniXcoder(
      (model): RobertaModel(
        (embeddings): RobertaEmbeddings(
          (word_embeddings): Embedding(51416, 768, padding_idx=1)
          (position_embeddings): Embedding(1026, 768, padding_idx=1)
          (token_type_embeddings): Embedding(10, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): RobertaEncoder(
          (layer): ModuleList(
            (0-11): 12 x RobertaLayer(
              (attention): RobertaAttention(
                (self): RobertaSdpaSelfAttention(
                  (query): Linear(in_features=768, out_features=768, bias=True)
                  (key): Linear(in_features=768, out_features=768, bias=True)
                  (value): Linear(in_features=768, out_features=768, bias=True)
                  (dropout): Dropout(p=0.1, inplace=False)
          

In [22]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Thu Jun 19 13:19:04 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.154.05             Driver Version: 535.154.05   CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA H100 80GB HBM3          On  | 00000000:19:00.0 Off |                    0 |
| N/A   44C    P0             456W / 700W |  60175MiB / 81559MiB |    100%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--